<a href="https://colab.research.google.com/github/miriamamin1213-ux/ValidationSET/blob/main/HancockGPT2Gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# HANCOCK GPT-2 GENERATIVE MODEL
# ============================================================
# 1. IMPORT LIBRARIES, SET RANDOM SEED AND SELECT DEVICE
# ============================================================
import os, random, json, gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_curve,
    precision_recall_curve,
)

SEED = 42


def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. LOAD AND MERGE HANCOCK CLINICAL/PATHOLOGICAL DATA
# ============================================================
CLINICAL_FILE = "clinical_data.json"
PATHOLOGICAL_FILE = "pathological_data.json"

if not os.path.exists(CLINICAL_FILE):
    if gdown.download(
        id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
        output=CLINICAL_FILE,
        quiet=False,
    ) is None:
        raise FileNotFoundError("Could not download clinical_data.json.")

if not os.path.exists(PATHOLOGICAL_FILE):
    if gdown.download(
        id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
        output=PATHOLOGICAL_FILE,
        quiet=False,
    ) is None:
        raise FileNotFoundError("Could not download pathological_data.json.")


def load_json_table(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        for key in ["data", "records", "patients", "items"]:
            if key in obj and isinstance(obj[key], list):
                return pd.DataFrame(obj[key])
        try:
            return pd.DataFrame(obj)
        except ValueError:
            pass
    raise ValueError(f"Could not convert {path} to a table.")


clinical_df = load_json_table(CLINICAL_FILE)
pathological_df = load_json_table(PATHOLOGICAL_FILE)

if (
    "patient_id" not in clinical_df.columns
    or "patient_id" not in pathological_df.columns
):
    raise KeyError("Both JSON files must contain 'patient_id'.")

df = clinical_df.merge(
    pathological_df,
    on="patient_id",
    how="left",
    suffixes=("", "_path"),
)


def resolve_col(candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"Required column not found: {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )
    return None


# ============================================================
# 3. SELECT SIX REQUIRED FEATURES AND DEFINE HPV TARGET
# ============================================================
target_col = resolve_col(
    ["hpv_association_p16", "hpv_association_p16_path"],
    required=True,
)
feature_candidates = {
    "age_at_initial_diagnosis": [
        "age_at_initial_diagnosis",
        "age_at_initial_diagnosis_path",
    ],
    "sex": ["sex", "sex_path"],
    "smoking_status": ["smoking_status", "smoking_status_path"],
    "primary_tumor_site": [
        "primary_tumor_site",
        "primary_tumor_site_path",
    ],
    "primarily_metastasis": [
        "primarily_metastasis",
        "primarily_metastasis_path",
    ],
    "pN_stage": [
        "pN_stage",
        "pN_stage_path",
        "pathological_N_stage",
        "pathological_n_stage",
    ],
}
feature_map = {
    name: resolve_col(cols, required=True)
    for name, cols in feature_candidates.items()
}
features = list(feature_candidates.keys())
df = df.dropna(subset=[target_col]).copy()


def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["negative", "neg", "0", "false", "no"]:
        return 0
    if s in ["positive", "pos", "1", "true", "yes"]:
        return 1
    try:
        n = float(s)
        if n in [0, 1]:
            return int(n)
    except ValueError:
        pass
    return np.nan


df["HPV_Label"] = df[target_col].apply(normalize_target)
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
feature_df = df[[feature_map[c] for c in features]].copy()
feature_df = feature_df.rename(
    columns={feature_map[c]: c for c in features}
)
df = pd.concat([feature_df, df[["HPV_Label"]]], axis=1)
df = df.loc[:, ~df.columns.duplicated()].copy()
df["age_at_initial_diagnosis"] = pd.to_numeric(
    df["age_at_initial_diagnosis"],
    errors="coerce",
)
print("Patients:", len(df))
print("HPV classes:\n", df["HPV_Label"].value_counts().sort_index())
print("Required six features:", features)
print("N-stage source column:", feature_map["pN_stage"])

# ============================================================
# 4. STRATIFIED TRAIN/VALIDATION/TEST SPLIT (65/15/20)
# ============================================================
train_val_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["HPV_Label"],
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1875,
    random_state=SEED,
    stratify=train_val_df["HPV_Label"],
)
train_df, val_df, test_df = (
    x.copy() for x in [train_df, val_df, test_df]
)

# ============================================================
# 5. HANDLE MISSING VALUES USING TRAINING DATA ONLY
# ============================================================
fill_values = {}
for c in features:
    if pd.api.types.is_numeric_dtype(train_df[c]):
        value = train_df[c].median()
        if pd.isna(value):
            value = 0
    else:
        modes = train_df[c].dropna().astype(str).mode()
        value = modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[c] = value
    for split in [train_df, val_df, test_df]:
        split[c] = split[c].fillna(value)
        if split[c].dtype == object:
            split[c] = split[c].astype(str).replace(
                {"": "Unknown", "nan": "Unknown", "None": "Unknown"}
            )

y_train = train_df["HPV_Label"].to_numpy(dtype=int)
y_val = val_df["HPV_Label"].to_numpy(dtype=int)
y_test = test_df["HPV_Label"].to_numpy(dtype=int)

print(
    "Train:",
    len(train_df),
    "classes:",
    np.bincount(y_train, minlength=2),
)
print(
    "Validation:",
    len(val_df),
    "classes:",
    np.bincount(y_val, minlength=2),
)
print(
    "Test:",
    len(test_df),
    "classes:",
    np.bincount(y_test, minlength=2),
)

# ============================================================
# 6. CONVERT SIX FEATURES INTO CLINICAL TEXT PROMPTS
# ============================================================
def display_value(x):
    if pd.isna(x):
        return "Unknown"
    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x).strip()


def patient_to_text(row):
    parts = [
        "Q: Predict this patient's Human Papillomavirus (HPV) status "
        "using the clinical information available."
    ]
    labels = {
        "age_at_initial_diagnosis": "Age at initial diagnosis",
        "sex": "Sex",
        "smoking_status": "Smoking status",
        "primary_tumor_site": "Primary tumour site",
        "primarily_metastasis": "Primarily metastasis",
        "pN_stage": "Pathological N stage",
    }
    for c in features:
        parts.append(f"{labels[c]}: {display_value(row[c])}.")
    parts.append(
        "Is this patient HPV positive? Answer with only Yes or No.\nA:"
    )
    return " ".join(parts)


train_texts = [
    patient_to_text(r) for _, r in train_df.iterrows()
]
val_texts = [
    patient_to_text(r) for _, r in val_df.iterrows()
]
test_texts = [
    patient_to_text(r) for _, r in test_df.iterrows()
]
train_targets = ["Yes" if y == 1 else "No" for y in y_train]
val_targets = ["Yes" if y == 1 else "No" for y in y_val]
test_targets = ["Yes" if y == 1 else "No" for y in y_test]

print("Example prompt:\n", train_texts[0], "\nTarget:", train_targets[0])

# ============================================================
# 7. LOAD TOKENIZER AND SET MAXIMUM SEQUENCE LENGTH
# ============================================================
MODEL_NAME = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

TRAIN_TARGET_LENGTHS = [
    len(tokenizer.encode(" " + t, add_special_tokens=False)) + 1
    for t in train_targets
]
TRAIN_PROMPT_LENGTHS = [
    len(tokenizer.encode(t, add_special_tokens=False))
    for t in train_texts
]
TRAIN_REQUIRED_LENGTHS = [
    p + t
    for p, t in zip(TRAIN_PROMPT_LENGTHS, TRAIN_TARGET_LENGTHS)
]

print("\n===== TOKEN LENGTH CHECK =====")
print(
    "Training prompt tokens: min =",
    min(TRAIN_PROMPT_LENGTHS),
    "| median =",
    int(np.median(TRAIN_PROMPT_LENGTHS)),
    "| 95th percentile =",
    int(np.percentile(TRAIN_PROMPT_LENGTHS, 95)),
    "| max =",
    max(TRAIN_PROMPT_LENGTHS),
)
print(
    "Training prompt + answer tokens: max =",
    max(TRAIN_REQUIRED_LENGTHS),
)
MAX_LENGTH = min(
    1024,
    max(32, int(np.ceil(max(TRAIN_REQUIRED_LENGTHS) / 8) * 8)),
)
if max(TRAIN_REQUIRED_LENGTHS) > 1024:
    print(
        "WARNING: Some examples exceed GPT-2's 1024-token context "
        "window and will be truncated."
    )
print("Selected MAX_LENGTH:", MAX_LENGTH)

BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
print("Batch size:", BATCH_SIZE, "| Epochs:", EPOCHS)

# ============================================================
# 8. BUILD TOKENIZED DATASETS AND DATA LOADERS
# ============================================================
class GPT2GenerationDataset(Dataset):
    def __init__(self, texts, targets, labels):
        self.items = []
        for text, target, label in zip(texts, targets, labels):
            prompt_ids = tokenizer.encode(
                text,
                add_special_tokens=False,
            )
            target_ids = tokenizer.encode(
                " " + target,
                add_special_tokens=False,
            )
            target_ids += [tokenizer.eos_token_id]
            prompt_ids = prompt_ids[:max(0, MAX_LENGTH - len(target_ids))]
            ids = prompt_ids + target_ids
            loss_labels = [-100] * len(prompt_ids) + target_ids
            pad_len = MAX_LENGTH - len(ids)
            self.items.append(
                (
                    ids + [tokenizer.pad_token_id] * pad_len,
                    [1] * len(ids) + [0] * pad_len,
                    loss_labels + [-100] * pad_len,
                    int(label),
                )
            )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ids, mask, labels, label = self.items[i]
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "class_label": torch.tensor(label, dtype=torch.long),
        }


train_dataset = GPT2GenerationDataset(
    train_texts, train_targets, y_train
)
val_dataset = GPT2GenerationDataset(val_texts, val_targets, y_val)
test_dataset = GPT2GenerationDataset(
    test_texts, test_targets, y_test
)

train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

# ============================================================
# 9. INITIALISE GPT-2, CLASS WEIGHTS AND ADAMW OPTIMIZER
# ============================================================
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

# Freeze all GPT-2 backbone and language-model head parameters
for param in model.parameters():
    param.requires_grad = False

model.config.pad_token_id = tokenizer.pad_token_id

# Verify that all parameters are frozen
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

assert trainable_params == 0, "Some GPT-2 parameters are still trainable!"

class_counts = np.bincount(y_train, minlength=2)
class_weights = torch.tensor(
    len(y_train) / (2 * np.maximum(class_counts, 1)),
    dtype=torch.float32,
    device=device,
)
# No optimizer is required because the entire generative model is frozen
optimizer = None

print("Class weights:", class_weights.detach().cpu().numpy())

# ============================================================
# 10. DEFINE CLASS-WEIGHTED GENERATION LOSS
# ============================================================
def weighted_generation_loss(logits, labels, class_labels):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    token_loss = nn.CrossEntropyLoss(reduction="none")(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
    ).view(shift_labels.size(0), -1)
    valid = (shift_labels != -100).float()
    example_loss = (token_loss * valid).sum(1) / valid.sum(1).clamp(
        min=1
    )
    return (example_loss * class_weights[class_labels]).mean()


def evaluate_loss(loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            classes = batch["class_label"].to(device)
            logits = model(
                input_ids=ids,
                attention_mask=mask,
            ).logits
            loss = weighted_generation_loss(logits, labels, classes)
            n = ids.size(0)
            total += loss.item() * n
            count += n
    return total / max(count, 1)


# ============================================================
# 11. CALCULATE HPV SCORES AND SELECT VALIDATION THRESHOLD
# ============================================================
yes_ids = tokenizer.encode(" Yes", add_special_tokens=False)
no_ids = tokenizer.encode(" No", add_special_tokens=False)

if len(yes_ids) != 1 or len(no_ids) != 1:
    raise ValueError("' Yes' and ' No' must each be a single GPT-2 token.")

YES_TOKEN_ID, NO_TOKEN_ID = yes_ids[0], no_ids[0]


def get_hpv_scores(prompts, batch_size=BATCH_SIZE):
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(prompts), batch_size):
            batch_prompts = prompts[start:start + batch_size]
            encoded = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(device)
            logits = model(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
            ).logits
            last_positions = encoded["attention_mask"].sum(dim=1) - 1
            next_logits = logits[
                torch.arange(len(batch_prompts), device=device),
                last_positions,
            ]
            probs = torch.softmax(next_logits, dim=-1)
            yes_prob = probs[:, YES_TOKEN_ID]
            no_prob = probs[:, NO_TOKEN_ID]
            normalized = yes_prob / (
                yes_prob + no_prob
            ).clamp(min=1e-12)
            scores.extend(normalized.cpu().numpy())
    return np.asarray(scores, dtype=float)


def select_threshold(scores, labels):
    thresholds = np.unique(
        np.concatenate(([0.0], scores, [1.0]))
    )
    results = [
        (
            float(t),
            balanced_accuracy_score(
                labels,
                (scores >= t).astype(int),
            ),
        )
        for t in thresholds
    ]
    # Maximise validation balanced accuracy; ties go to threshold
    # closest to 0.5.
    return max(
        results,
        key=lambda x: (x[1], -abs(x[0] - 0.5)),
    )


# ============================================================
# 12. EVALUATE FROZEN GPT-2 ON VALIDATION DATA
# ============================================================

OUT_DIR = Path("ROC_HANCOCK_GPT2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

model.eval()

val_loss = evaluate_loss(val_loader)
val_scores = get_hpv_scores(val_texts)
val_auc = roc_auc_score(y_val, val_scores)
val_ap = average_precision_score(y_val, val_scores)
val_threshold, val_ba = select_threshold(val_scores, y_val)

print("Frozen GPT-2 validation results")
print("Validation Loss:", val_loss)
print("Validation ROC-AUC:", val_auc)
print("Validation PR-AUC:", val_ap)
print("Validation Balanced Accuracy:", val_ba)
print("Validation Threshold:", val_threshold)

best = {
    "Frozen_GPT2": {
        "value": val_ba,
        "epoch": 0,
        "threshold": val_threshold,
    }
}

checkpoint_paths = {
    "Frozen_GPT2": OUT_DIR / "frozen_gpt2.pth"
}

torch.save(model.state_dict(), checkpoint_paths["Frozen_GPT2"])

best_summary = pd.DataFrame([{
    "Checkpoint_Selected_By": "Frozen_GPT2",
    "Best_Epoch": 0,
    "Best_Validation_Metric": val_ba,
    "Threshold_Selected_On_Validation": val_threshold,
    "Checkpoint_Path": str(checkpoint_paths["Frozen_GPT2"]),
}])

best_summary.to_csv(
    OUT_DIR / "best_validation_checkpoints.csv",
    index=False,
)
# ============================================================
# 13. EVALUATE FOUR CHECKPOINTS ON HELD-OUT TEST SET
# ============================================================
test_results, test_predictions = [], {}

for metric, path in checkpoint_paths.items():
    print(f"\n===== CHECKPOINT: {metric} =====")
    model.eval()
    scores = get_hpv_scores(test_texts)
    threshold = best[metric]["threshold"]
    pred = (scores >= threshold).astype(int)

    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    ba = balanced_accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, zero_division=0)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])

    print("Best epoch:", best[metric]["epoch"])
    print(f"Validation threshold: {threshold:.6f}")
    print(
        classification_report(
            y_test,
            pred,
            labels=[0, 1],
            target_names=["HPV Negative", "HPV Positive"],
            digits=4,
            zero_division=0,
        )
    )
    print(
        f"Test ROC-AUC: {auc:.4f} | "
        f"Test PR-AUC (AP): {ap:.4f} | "
        f"Test Balanced Accuracy: {ba:.4f} | F1: {f1:.4f}"
    )
    print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)

    test_results.append({
        "Dataset": "HANCOCK",
        "Model": "GPT-2 Generative",
        "Checkpoint_Selected_By": metric,
        "Best_Epoch": best[metric]["epoch"],
        "Best_Validation_Metric": best[metric]["value"],
        "Threshold_Selected_On_Validation": threshold,
        "Test_ROC_AUC": auc,
        "Test_PR_AUC_Average_Precision": ap,
        "Test_Balanced_Accuracy": ba,
        "Test_F1": f1,
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
    })

    test_predictions[metric] = {
        "scores": scores,
        "predictions": pred,
        "labels": y_test,
        "threshold": threshold,
    }

    pd.DataFrame({
        "True_Label": y_test,
        "HPV_Positive_Score": scores,
        "Predicted_Label": pred,
    }).to_csv(
        OUT_DIR / f"test_predictions_{metric.lower()}.csv",
        index=False,
    )

results_df = pd.DataFrame(test_results)
results_df.to_csv(
    OUT_DIR / "test_metrics_by_checkpoint.csv",
    index=False,
)
print(
    "\nTEST RESULTS BY CHECKPOINT\n",
    results_df.to_string(index=False),
)

# ============================================================
# 14. SAVE ROC AND PRECISION-RECALL CURVES FOR ALL CHECKPOINTS
# ============================================================
for metric, result in test_predictions.items():
    labels = result["labels"]
    scores = result["scores"]
    auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    precision, recall, _ = precision_recall_curve(labels, scores)
    safe_name = metric.lower()

    pd.DataFrame({
        "FPR": fpr,
        "TPR": tpr,
        "Threshold": roc_thresholds,
    }).to_csv(
        OUT_DIR / f"roc_coordinates_{safe_name}.csv",
        index=False,
    )
    pd.DataFrame({
        "Precision": precision,
        "Recall": recall,
    }).to_csv(
        OUT_DIR / f"precision_recall_coordinates_{safe_name}.csv",
        index=False,
    )

    plt.figure(figsize=(7, 6))
    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"ROC-AUC = {auc:.4f}",
    )
    plt.plot(
        [0, 1],
        [0, 1],
        "--",
        linewidth=1.5,
        label="Random",
    )
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"HANCOCK GPT-2 Generative ROC: {metric}")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"roc_curve_{safe_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

    plt.figure(figsize=(7, 6))
    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=f"PR-AUC (AP) = {ap:.4f}",
    )
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(
        f"HANCOCK GPT-2 Generative Precision-Recall: {metric}"
    )
    plt.legend(loc="lower left")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"precision_recall_curve_{safe_name}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

# ============================================================
# 15. SAVE SETTINGS AND MODEL EVALUATION OUTPUTS
# ============================================================
settings = {
    "Dataset": "HANCOCK",
    "Model": MODEL_NAME,
    "Features": ", ".join(features),
    "Number_of_Features": len(features),
    "Target": target_col,
    "Learning_Rate": LEARNING_RATE,
    "Weight_Decay": WEIGHT_DECAY,
    "Optimizer": "None - fully frozen model",
    "Epochs": 0,
    "Batch_Size": BATCH_SIZE,
    "Max_Length": MAX_LENGTH,
    "Random_Seed": SEED,
    "Checkpoint_Selection": (
        "Best validation loss, ROC-AUC, PR-AUC "
        "and balanced accuracy"
    ),
    "Threshold_Selection": (
        "Maximise validation balanced accuracy; ties closest to 0.5"
    ),
    "F1": "Reported at the validation-selected threshold",
    "Curves": (
        "Test ROC and precision-recall curves saved "
        "for each checkpoint"
    ),
}
pd.DataFrame([settings]).to_csv(
    OUT_DIR / "settings.csv",
    index=False,
)
print("\nFeatures used:", features)
print("All HANCOCK results saved to:", OUT_DIR.resolve())

# ============================================================
# 16. APPEND HANCOCK AUC RESULTS TO HECKTOR COMPARISON CSV
# ============================================================
comparison_file = Path("model_roc_pr_auc_comparison.csv")
hancock_results = results_df[
    [
        "Dataset",
        "Model",
        "Checkpoint_Selected_By",
        "Best_Epoch",
        "Test_ROC_AUC",
        "Test_PR_AUC_Average_Precision",
        "Test_Balanced_Accuracy",
        "Test_F1",
    ]
].copy()
hancock_results = hancock_results.rename(
    columns={"Test_PR_AUC_Average_Precision": "Test_PR_AUC"}
)

if os.path.exists(comparison_file):
    existing = pd.read_csv(comparison_file)
    for col in hancock_results.columns:
        if col not in existing.columns:
            existing[col] = np.nan
    for col in existing.columns:
        if col not in hancock_results.columns:
            hancock_results[col] = np.nan
    combined = pd.concat(
        [existing, hancock_results[existing.columns]],
        ignore_index=True,
    )
    combined = combined.drop_duplicates(
        subset=["Dataset", "Model", "Checkpoint_Selected_By"],
        keep="last",
    )
else:
    combined = hancock_results

combined.to_csv(comparison_file, index=False)
print("AUC comparison saved to:", comparison_file)

Device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 27.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 7.74MB/s]


Patients: 332
HPV classes:
 HPV_Label
0    191
1    141
Name: count, dtype: int64
Required six features: ['age_at_initial_diagnosis', 'sex', 'smoking_status', 'primary_tumor_site', 'primarily_metastasis', 'pN_stage']
N-stage source column: pN_stage
Train: 215 classes: [123  92]
Validation: 50 classes: [29 21]
Test: 67 classes: [39 28]
Example prompt:
 Q: Predict this patient's Human Papillomavirus (HPV) status using the clinical information available. Age at initial diagnosis: 53. Sex: male. Smoking status: smoker. Primary tumour site: Oropharynx. Primarily metastasis: no. Pathological N stage: pN0. Is this patient HPV positive? Answer with only Yes or No.
A: 
Target: No


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Trainable parameters: 0
Total parameters: 124439808
Class weights: [0.87398374 1.1684783 ]
Frozen GPT-2 validation results
Validation Loss: 5.8688645362854
Validation ROC-AUC: 0.5689655172413792
Validation PR-AUC: 0.4667515712150291
Validation Balanced Accuracy: 0.5853858784893268
Validation Threshold: 0.47603538632392883

===== CHECKPOINT: Frozen_GPT2 =====
Best epoch: 0
Validation threshold: 0.476035
              precision    recall  f1-score   support

HPV Negative     0.6000    0.4615    0.5217        39
HPV Positive     0.4324    0.5714    0.4923        28

    accuracy                         0.5075        67
   macro avg     0.5162    0.5165    0.5070        67
weighted avg     0.5300    0.5075    0.5094        67

Test ROC-AUC: 0.5009 | Test PR-AUC (AP): 0.4198 | Test Balanced Accuracy: 0.5165 | F1: 0.4923
Confusion matrix [[TN, FP], [FN, TP]]:
 [[18 21]
 [12 16]]

TEST RESULTS BY CHECKPOINT
 Dataset            Model Checkpoint_Selected_By  Best_Epoch  Best_Validation_Metric  

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# Google Drive
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = Path("/content/drive/MyDrive/HANCOCK_GPT2_Generative_Frozen_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Check the existing results from your completed run
print("Available result variables:")
for name in ["test_results", "test_predictions", "best", "checkpoint_paths"]:
    print(name, ":", name in globals())

# Save existing test metrics
if "test_results" in globals():
    pd.DataFrame(test_results).to_csv(
        OUT_DIR / "test_metrics_frozen_gpt2.csv", index=False
    )

# Save existing test predictions
if "test_predictions" in globals():
    for metric, result in test_predictions.items():
        scores = np.asarray(result["scores"])
        predictions = np.asarray(result["predictions"])
        labels = np.asarray(result["labels"])

        pd.DataFrame({
            "True_Label": labels,
            "HPV_Positive_Score": scores,
            "Predicted_Label": predictions,
            "Threshold": result["threshold"]
        }).to_csv(
            OUT_DIR / f"test_predictions_{metric.lower()}.csv",
            index=False
        )

# Copy existing output files and plots from the original run folder
SOURCE_DIR = Path("/content/ROC_HANCOCK_GPT2")

if SOURCE_DIR.exists():
    for file in SOURCE_DIR.iterdir():
        if file.is_file():
            shutil.copy2(file, OUT_DIR / file.name)
    print("\nCopied existing files from:", SOURCE_DIR)
else:
    print("\nOriginal output folder not found:", SOURCE_DIR)

# Verify saved files
print("\n========== DRIVE FILE CHECK ==========")
for file in sorted(OUT_DIR.iterdir()):
    print(file.name, "-", file.stat().st_size, "bytes")

# Verify prediction rows if available
print("\n========== PREDICTION CHECK ==========")
prediction_files = list(OUT_DIR.glob("test_predictions_*.csv"))

for file in prediction_files:
    df = pd.read_csv(file)
    print(file.name, "| rows:", len(df),
          "| classes:", df["True_Label"].value_counts().to_dict())

print("\nSaved folder:", OUT_DIR)

Mounted at /content/drive
Available result variables:
test_results : True
test_predictions : True
best : True
checkpoint_paths : True

Copied existing files from: /content/ROC_HANCOCK_GPT2

========== DRIVE FILE CHECK ==========
best_validation_checkpoints.csv - 192 bytes
frozen_gpt2.pth - 497815679 bytes
precision_recall_coordinates_frozen_gpt2.csv - 2109 bytes
precision_recall_curve_frozen_gpt2.png - 120299 bytes
roc_coordinates_frozen_gpt2.csv - 1949 bytes
roc_curve_frozen_gpt2.png - 129600 bytes
settings.csv - 610 bytes
test_metrics_by_checkpoint.csv - 358 bytes
test_metrics_frozen_gpt2.csv - 358 bytes
test_predictions_frozen_gpt2.csv - 1604 bytes

========== PREDICTION CHECK ==========
test_predictions_frozen_gpt2.csv | rows: 67 | classes: {0: 39, 1: 28}

Saved folder: /content/drive/MyDrive/HANCOCK_GPT2_Generative_Frozen_results


In [ ]:
import os
import pandas as pd

RESULTS_DIR = "/content/ROC_HANCOCK_GPT2"

print("Folder exists:", os.path.exists(RESULTS_DIR))

if os.path.exists(RESULTS_DIR):
    print("\nFiles saved:")
    for f in sorted(os.listdir(RESULTS_DIR)):
        print(f)

    print("\nChecking test metrics:")
    metrics_file = os.path.join(
        RESULTS_DIR, "test_metrics_by_checkpoint.csv"
    )

    if os.path.exists(metrics_file):
        display(pd.read_csv(metrics_file))
    else:
        print("Test metrics CSV not found.")

    print("\nChecking test predictions:")
    for f in os.listdir(RESULTS_DIR):
        if "test_predictions" in f.lower():
            path = os.path.join(RESULTS_DIR, f)
            df = pd.read_csv(path)
            print(f"{f}: {len(df)} rows")
            display(df.head())

Folder exists: True

Files saved:
best_validation_checkpoints.csv
frozen_gpt2.pth
precision_recall_coordinates_frozen_gpt2.csv
precision_recall_curve_frozen_gpt2.png
roc_coordinates_frozen_gpt2.csv
roc_curve_frozen_gpt2.png
settings.csv
test_metrics_by_checkpoint.csv
test_predictions_frozen_gpt2.csv

Checking test metrics:


,Dataset,Model,Checkpoint_Selected_By,Best_Epoch,Best_Validation_Metric,Threshold_Selected_On_Validation,Test_ROC_AUC,Test_PR_AUC_Average_Precision,Test_Balanced_Accuracy,Test_F1,TN,FP,FN,TP
0,HANCOCK,GPT-2 Generative,Frozen_GPT2,0,0.585386,0.476035,0.500916,0.419789,0.516484,0.492308,18,21,12,16



Checking test predictions:
test_predictions_frozen_gpt2.csv: 67 rows


,True_Label,HPV_Positive_Score,Predicted_Label
0,1,0.472026,0
1,1,0.473433,0
2,0,0.462367,0
3,1,0.489679,1
4,1,0.501358,1


In [ ]:

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    recall_score,
    precision_score,
    f1_score
)

# =====================================================
# 1. LOAD SAVED HELD-OUT TEST PREDICTIONS
# =====================================================

RESULTS_DIR = "/content/ROC_HANCOCK_GPT2"

prediction_file = os.path.join(
    RESULTS_DIR, "test_predictions_frozen_gpt2.csv"
)

df = pd.read_csv(prediction_file)

y_true = df["True_Label"].astype(int).to_numpy()
y_score = df["HPV_Positive_Score"].astype(float).to_numpy()
y_pred = df["Predicted_Label"].astype(int).to_numpy()

print("Test patients:", len(df))
print("Negative:", np.sum(y_true == 0))
print("Positive:", np.sum(y_true == 1))

# =====================================================
# 2. DEFINE METRICS
# =====================================================

def calculate_metrics(y, scores, predictions):

    return {
        "ROC_AUC": roc_auc_score(y, scores),
        "PR_AUC_Average_Precision": average_precision_score(y, scores),
        "Balanced_Accuracy": balanced_accuracy_score(y, predictions),
        "Sensitivity": recall_score(y, predictions, zero_division=0),
        "Specificity": recall_score(
            y, predictions, pos_label=0, zero_division=0
        ),
        "F1": f1_score(y, predictions, zero_division=0)
    }

# Original test estimates
original_metrics = calculate_metrics(
    y_true, y_score, y_pred
)

# =====================================================
# 3. STRATIFIED BOOTSTRAP
# =====================================================

N_BOOTSTRAP = 5000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

negative_indices = np.where(y_true == 0)[0]
positive_indices = np.where(y_true == 1)[0]

bootstrap_results = {
    metric: [] for metric in original_metrics
}

for b in range(N_BOOTSTRAP):

    # Resample each class separately, preserving class counts
    sampled_negative = rng.choice(
        negative_indices,
        size=len(negative_indices),
        replace=True
    )

    sampled_positive = rng.choice(
        positive_indices,
        size=len(positive_indices),
        replace=True
    )

    sampled_indices = np.concatenate([
        sampled_negative, sampled_positive
    ])

    y_boot = y_true[sampled_indices]
    scores_boot = y_score[sampled_indices]
    pred_boot = y_pred[sampled_indices]

    metrics = calculate_metrics(
        y_boot, scores_boot, pred_boot
    )

    for metric, value in metrics.items():
        bootstrap_results[metric].append(value)

# =====================================================
# 4. CALCULATE 95% CONFIDENCE INTERVALS
# =====================================================

rows = []

for metric, estimate in original_metrics.items():

    values = np.asarray(bootstrap_results[metric])

    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)

    rows.append({
        "Dataset": "HANCOCK",
        "Model": "GPT-2 Generative (Fully Frozen)",
        "Metric": metric,
        "Test_Estimate": estimate,
        "CI_95_Lower": lower,
        "CI_95_Upper": upper,
        "Bootstrap_Replicates": N_BOOTSTRAP
    })

ci_df = pd.DataFrame(rows)

# =====================================================
# 5. SAVE RESULTS TO GOOGLE DRIVE
# =====================================================

output_file = os.path.join(
    RESULTS_DIR,
    "HANCOCK_GPT2_Generative_Frozen_Confidence_Intervals.csv"
)

ci_df.to_csv(output_file, index=False)

print("\n===== 95% BOOTSTRAP CONFIDENCE INTERVALS =====")
display(ci_df.round(4))

print("\nSaved successfully:")
print(output_file)


Test patients: 67
Negative: 39
Positive: 28

===== 95% BOOTSTRAP CONFIDENCE INTERVALS =====


,Dataset,Model,Metric,Test_Estimate,CI_95_Lower,CI_95_Upper,Bootstrap_Replicates
0,HANCOCK,GPT-2 Generative (Fully Frozen),ROC_AUC,0.5009,0.3590,0.6438,5000
1,HANCOCK,GPT-2 Generative (Fully Frozen),PR_AUC_Average_Precision,0.4198,0.3539,0.5623,5000
2,HANCOCK,GPT-2 Generative (Fully Frozen),Balanced_Accuracy,0.5165,0.3960,0.6342,5000
3,HANCOCK,GPT-2 Generative (Fully Frozen),Sensitivity,0.5714,0.3929,0.7500,5000
4,HANCOCK,GPT-2 Generative (Fully Frozen),Specificity,0.4615,0.3077,0.6154,5000
5,HANCOCK,GPT-2 Generative (Fully Frozen),F1,0.4923,0.3548,0.6129,5000



Saved successfully:
/content/ROC_HANCOCK_GPT2/HANCOCK_GPT2_Generative_Frozen_Confidence_Intervals.csv


In [1]:
print("Checking Hancock GPT-2 runtime...")

print("Model exists:", "model" in globals())

if "model" in globals():
    print("Model is loaded.")

    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    total = sum(
        p.numel() for p in model.parameters()
    )

    print("Trainable parameters:", trainable)
    print("Total parameters:", total)
    print("Model device:", next(model.parameters()).device)
else:
    print("Model is not currently loaded.")

Checking Hancock GPT-2 runtime...
Model exists: False
Model is not currently loaded.


In [2]:
import os

print("Checking Hancock notebook files...")

for root in ["/content", "/content/drive/MyDrive"]:
    print(f"\nSearching {root}")

    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in [
            ".config", "sample_data", ".cache"
        ]]

        for filename in files:
            name = filename.lower()

            if "hancock" in name or (
                "frozen_gpt2" in name
            ):
                print(os.path.join(current, filename))

Checking Hancock notebook files...

Searching /content

Searching /content/drive/MyDrive


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
MODEL_NAME = "openai-community/gpt2"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Model:", MODEL_NAME)
print("Device:", device)

Model: openai-community/gpt2
Device: cpu


In [5]:
# ============================================================
# RECOVER AND SAVE HANCOCK FROZEN GPT-2 RESULTS TO GOOGLE DRIVE
# NO TRAINING
# ============================================================

import os
import json
import shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from transformers import GPT2LMHeadModel
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_curve,
    precision_recall_curve
)

# 1. Google Drive must already be mounted
DRIVE_DIR = Path("/content/drive/MyDrive")
assert DRIVE_DIR.exists(), "Google Drive is not mounted."

OUT_DIR = DRIVE_DIR / "HANCOCK_GPT2_Generative_Frozen_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Rebuild the same fully frozen pretrained GPT-2
recovery_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

for param in recovery_model.parameters():
    param.requires_grad = False

recovery_model.eval()

assert sum(p.numel() for p in recovery_model.parameters()
           if p.requires_grad) == 0

# Use the existing scoring function with the rebuilt model
original_model = model
model = recovery_model

# 3. Recalculate validation scores and threshold
val_scores_recovered = get_hpv_scores(val_texts)

val_threshold_recovered, val_ba_recovered = select_threshold(
    val_scores_recovered, y_val
)

# 4. Calculate held-out test predictions
test_scores_recovered = get_hpv_scores(test_texts)

test_predictions_recovered = (
    test_scores_recovered >= val_threshold_recovered
).astype(int)

# 5. Calculate test metrics
test_roc = roc_auc_score(y_test, test_scores_recovered)
test_pr = average_precision_score(y_test, test_scores_recovered)
test_ba = balanced_accuracy_score(y_test, test_predictions_recovered)
test_f1 = f1_score(y_test, test_predictions_recovered, zero_division=0)

cm = confusion_matrix(
    y_test, test_predictions_recovered, labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("Hancock frozen GPT-2 recovered results")
print("Test patients:", len(y_test))
print("Validation threshold:", val_threshold_recovered)
print("ROC-AUC:", test_roc)
print("PR-AUC:", test_pr)
print("Balanced accuracy:", test_ba)
print("F1:", test_f1)
print("Confusion matrix:\n", cm)

# 6. Save test predictions
pd.DataFrame({
    "True_Label": y_test,
    "HPV_Positive_Score": test_scores_recovered,
    "Predicted_Label": test_predictions_recovered
}).to_csv(
    OUT_DIR / "test_predictions_frozen_gpt2.csv",
    index=False
)

# 7. Save ROC coordinates and plot
fpr, tpr, roc_thresholds = roc_curve(
    y_test, test_scores_recovered
)

pd.DataFrame({
    "FPR": fpr,
    "TPR": tpr,
    "Threshold": roc_thresholds
}).to_csv(
    OUT_DIR / "roc_coordinates_frozen_gpt2.csv",
    index=False
)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {test_roc:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Hancock Frozen GPT-2 Generative ROC")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_curve_frozen_gpt2.png",
            dpi=300, bbox_inches="tight")
plt.close()

# 8. Save PR coordinates and plot
precision, recall, _ = precision_recall_curve(
    y_test, test_scores_recovered
)

pd.DataFrame({
    "Precision": precision,
    "Recall": recall
}).to_csv(
    OUT_DIR / "precision_recall_coordinates_frozen_gpt2.csv",
    index=False
)

plt.figure(figsize=(7, 6))
plt.plot(recall, precision, label=f"PR-AUC (AP) = {test_pr:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Hancock Frozen GPT-2 Generative Precision-Recall")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "precision_recall_curve_frozen_gpt2.png",
            dpi=300, bbox_inches="tight")
plt.close()

# 9. Save metrics and settings
pd.DataFrame([{
    "Dataset": "HANCOCK",
    "Model": "GPT-2 Generative Fully Frozen",
    "Test_Patients": len(y_test),
    "Validation_Threshold": val_threshold_recovered,
    "Validation_Balanced_Accuracy": val_ba_recovered,
    "Test_ROC_AUC": test_roc,
    "Test_PR_AUC": test_pr,
    "Test_Balanced_Accuracy": test_ba,
    "Test_F1": test_f1,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
    "Trainable_Parameters": 0
}]).to_csv(
    OUT_DIR / "test_metrics_frozen_gpt2.csv",
    index=False
)

pd.DataFrame([{
    "Dataset": "HANCOCK",
    "Model": MODEL_NAME,
    "Frozen_Backbone": True,
    "Trainable_Parameters": 0,
    "Epochs": 0,
    "Random_Seed": SEED,
    "Threshold_Selection": "Validation balanced accuracy",
    "Features": ", ".join(features)
}]).to_csv(
    OUT_DIR / "settings.csv",
    index=False
)

# 10. Restore original model variable
model = original_model

print("\nAll Hancock frozen GPT-2 results saved to:")
print(OUT_DIR)

print("\nSaved files:")
for file in sorted(OUT_DIR.iterdir()):
    print(file.name)

NameError: name 'MODEL_NAME' is not defined